# ForgeAI M2 CPU-only compiler validation (not executed)

Maintainer approval required before any cell execution. Settings: **Accelerator: None; Internet: Off**.
Attach only the locally validated transfer bundle as a private input. No Gemma weights, credentials,
provider inputs or evaluator tasks. Do not run the official starter notebook or its installation cell.
This notebook contains no package installation. If installed metadata fails, STOP for separate preparation approval.
Approval covers environment inspection, official offline directory validation/full compilation with inert factories,
and ZIP evidence writing only. It does not authorize model execution or competition submission.


In [ ]:
# Operator must set this only after separate explicit approval and checking notebook settings.
AUTHORIZATION = ""
EXPECTED = "AUTHORIZE FORGEAI GEMMA4 M2 KAGGLE CPU COMPILER ONLY"
if AUTHORIZATION != EXPECTED:
    raise RuntimeError("Compiler-only notebook not authorized")

import importlib.metadata as metadata
import json
from pathlib import Path
import sys

if sys.version_info[:2] != (3, 12) or sys.platform != "linux":
    raise RuntimeError("Requires separately prepared CPython 3.12 Linux environment")
# Adjust only to the actual attached bundle root; preserve all relative paths/bytes.
BUNDLE = Path("/kaggle/input/forgeai-gemma4-m2/bundle")
EXPECTED_PACKAGES = {"adk-submission": "0.2.13", "adk-eval-core": "0.1.0",
                     "swegemma": "0.2.11", "google-adk": "1.36.1", "google-genai": "2.11.0"}
observed = {}
for name in EXPECTED_PACKAGES:
    try:
        observed[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        observed[name] = "MISSING"
print(json.dumps({"python": sys.version, "platform": sys.platform, "packages": observed}, indent=2))
if observed != EXPECTED_PACKAGES:
    raise RuntimeError("KAGGLE_COMPILER_ENVIRONMENT_PREPARATION_REQUIRED; no installation authorized")
if not (BUNDLE / "local-receipt.json").is_file():
    raise RuntimeError("Missing byte-bound local scope/static receipt")


## Offline compilation and packaging
The portable gate verifies the local receipt and input bytes, checks pinned wheel and installed source bytes,
then runs the same official APIs and negative controls as the local gate in a guarded child process.
No tool execution, serving process, model inference, Docker or evaluator is invoked.
Imports may still fail because of runtime dependencies; that is a blocker, never permission to install.


In [ ]:
import subprocess
import tempfile

OUTPUT = Path("/kaggle/working/gemma4-m2-compiler")
LOG = Path("/kaggle/working/gemma4-m2-compiler.log")
if OUTPUT.exists() or LOG.exists():
    raise RuntimeError("Preserve existing evidence; choose a new output/log path after review")
with tempfile.TemporaryDirectory(prefix="gemma-notebook-home-") as home:
    clean_env = {"HOME": home, "TMPDIR": home, "CUDA_VISIBLE_DEVICES": "",
                 "LITELLM_LOCAL_MODEL_COST_MAP": "True", "OTEL_SDK_DISABLED": "true"}
    with LOG.open("x", encoding="utf-8") as log:
        completed = subprocess.run(
            [sys.executable, "-I", "-B", str(BUNDLE / "experiments/gemma4_competition/portable_m2.py"),
             "--receipt", str(BUNDLE / "local-receipt.json"), "--output", str(OUTPUT)],
            cwd=home, env=clean_env, stdout=log, stderr=subprocess.STDOUT, timeout=300,
        )
print(LOG.read_text(encoding="utf-8"))
if completed.returncode:
    raise RuntimeError("Portable compiler gate failed/blocked; preserve log and STOP")


## Downloadable evidence — no submission
Download `gemma4-m2-compiler.log` and the entire `gemma4-m2-compiler/` directory:
`compiler-output.json`, `manifest.json`, `submission.zip`.
The manifest contains sorted member names/sizes/SHA256, ZIP SHA256, Python/platform/package versions,
wheel hashes and the local receipt hash. Preserve the original local receipt.
Return these files for local reconciliation and later evidence-commit review; do not submit the ZIP to Kaggle.
A portable pass alone does not emit combined M2 qualification.
